# 01 — Version-pinned Silero segmentation and visual QC

Runs Silero, creates the mandatory review queue, supports audio/manual boundary review, and freezes adjudicated segments.

This notebook saves its visual summary and audit tables into separate `figures/` and `tables/` directories. Its final cell states the main output, any decision required, and whether the next stage is allowed.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from IPython.display import Image, Markdown, display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the paper_1 project.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
MAIN_OUTPUTS = ROOT / "MAIN outputs"
MAIN_OUTPUTS.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def read_table(path_without_suffix):
    stem = Path(path_without_suffix)
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing table: {parquet} or {csv}")

def stage_directories(relative_stage):
    stage = OUTPUT / relative_stage
    figures = stage / "figures"
    tables = stage / "tables"
    figures.mkdir(parents=True, exist_ok=True)
    tables.mkdir(parents=True, exist_ok=True)
    return stage, figures, tables

def save_table(frame, directory, name):
    path = Path(directory) / f"{name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, directory, name):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    png = directory / f"{name}.png"
    svg = directory / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

def stage_gate(stage_name, can_continue, reasons, next_step):
    status = "PASS — safe to continue" if can_continue else "BLOCKED — decision/action required"
    color = "#1B7F3A" if can_continue else "#B22222"
    details = "\n".join(f"- {reason}" for reason in reasons) if reasons else "- No blocking findings."
    display(Markdown(
        f"### {stage_name}: <span style='color:{color}'>{status}</span>\n\n"
        f"{details}\n\n**Next step:** {next_step}"
    ))
    return can_continue

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("MAIN outputs:", MAIN_OUTPUTS)


## Main output and decisions

Authoritative outputs: versioned `frozen_segmentation_decisions.csv` and `frozen_segmentation_intervals.csv` under `MAIN outputs/01_SEGMENTATION_FREEZE/<version>/`. Audit outputs include `bamboo_segmentation_intervals.csv`, one original-format `_segments.csv` and `_frames.csv` per recording under `outputs/01_segmentation/segmentation/silero/`, and the original four-panel figures under `outputs/01_segmentation/figures/segmentation/silero/{accepted,flagged,excluded}`. A separate boundary-audit CSV and PNG compare sample-index analysis edges with the 30-ms display representation.

After review, `segment-adjudicate` creates a separate immutable publication tree at `outputs/01_segmentation_after_review/`, with per-recording figures, frames, segments, and boundary audits organized as accepted, flagged, or excluded. Accepted and flagged recordings proceed; excluded recordings do not.

The segment roles are exactly `leading_nonspeech`, `internal_nonspeech`, `trailing_nonspeech`, and `speech`; the figure rows are labelled `non-speech` and `speech`. The primary boundaries use zero speech padding and no second post-VAD bridge/filter pass; the 30-ms mask is visualization, not the frozen timing source.

Decision required: inspect every flagged/excluded recording and every accepted recording selected by the prespecified segmentation-only outlier rules. Listen to the audio when needed. Use `KEEP + AUTO`, `KEEP + MANUAL`, or `EXCLUDE + NONE`. Feature extraction is blocked until eligibility and the final primary speech boundaries are frozen.

In [ ]:
RUN_SEGMENTATION = False  # Change to True only when you intend to run this stage.

if RUN_SEGMENTATION:
    run_cli('segment')
else:
    print('Segmentation not run. Set RUN_SEGMENTATION=True only after the data freeze passes.')

In [ ]:
STAGE, FIGURES, TABLES = stage_directories("01_segmentation")
summary = read_table(STAGE / "bamboo_segmentation_summary")
status_counts = read_table(STAGE / "segmentation_qc_status_counts")
flag_counts = read_table(STAGE / "segmentation_qc_flag_counts")
display(status_counts)
display(flag_counts.head(20))

LEGACY = STAGE / "segmentation" / "silero"
LEGACY_FIGURES = STAGE / "figures" / "segmentation" / "silero"
legacy_summary = pd.read_csv(LEGACY / "summary" / "silero_all_summary.csv")

artifact_audit = pd.DataFrame([{
    "frozen_recordings": len(summary),
    "summary_rows": len(legacy_summary),
    "segment_csv_files": len(list((LEGACY / "segments").glob("*_segments.csv"))),
    "frame_csv_files": len(list((LEGACY / "frames").glob("*_frames.csv"))),
    "accepted_png_files": len(list((LEGACY_FIGURES / "accepted").glob("*_silero.png"))),
    "flagged_png_files": len(list((LEGACY_FIGURES / "flagged").glob("*_silero.png"))),
    "excluded_png_files": len(list((LEGACY_FIGURES / "excluded").glob("*_silero.png"))),
    "boundary_audit_csv_files": len(list((LEGACY / "boundary_audit").glob("*_boundary_audit.csv"))),
    "boundary_audit_png_files": len(list((LEGACY_FIGURES / "boundary_audit").glob("*_boundary_audit.png"))),
}])
artifact_audit["total_png_files"] = artifact_audit[
    ["accepted_png_files", "flagged_png_files", "excluded_png_files"]
].sum(axis=1)
display(artifact_audit)
save_table(artifact_audit, TABLES, "notebook_silero_artifact_audit")

expected = int(artifact_audit.iloc[0]["frozen_recordings"])
assert int(artifact_audit.iloc[0]["summary_rows"]) == expected
assert int(artifact_audit.iloc[0]["segment_csv_files"]) == expected
assert int(artifact_audit.iloc[0]["frame_csv_files"]) == expected
assert int(artifact_audit.iloc[0]["total_png_files"]) == expected
assert int(artifact_audit.iloc[0]["boundary_audit_csv_files"]) == expected
assert int(artifact_audit.iloc[0]["boundary_audit_png_files"]) == expected

print("Representative original-pipeline diagnostic figures:")
for status in ["accepted", "flagged", "excluded"]:
    subset = summary.loc[summary["qc_status"].eq(status)]
    if not subset.empty and Path(subset.iloc[0]["plot_path"]).exists():
        display(Markdown(f"**{status.upper()} example**"))
        display(Image(filename=subset.iloc[0]["plot_path"], width=1000))

In [ ]:
# Boundary science audit: exact timestamps are not the 30-ms display bins.
resolved_parameters = json.loads(
    (STAGE / "logs" / "silero_segmentation_config.json").read_text(encoding="utf-8")
)
display(pd.DataFrame([resolved_parameters]).T.rename(columns={0: "resolved_value"}))

boundary_summary = summary[[
    "file_name", "qc_status", "boundary_edges",
    "boundary_low_contrast_edges", "boundary_low_contrast_fraction",
    "boundary_min_contrast_db", "boundary_audit_path", "boundary_plot_path",
]].copy()
display(boundary_summary.sort_values(
    ["boundary_low_contrast_fraction", "boundary_min_contrast_db"],
    ascending=[False, True],
).head(30))
save_table(boundary_summary, TABLES, "notebook_boundary_alignment_summary")

EXAMPLE_BOUNDARY_FILE = None  # set exact filename, or use the most ambiguous edge record
if EXAMPLE_BOUNDARY_FILE:
    boundary_row = boundary_summary.loc[
        boundary_summary["file_name"].eq(EXAMPLE_BOUNDARY_FILE)
    ].iloc[0]
else:
    boundary_row = boundary_summary.sort_values(
        ["boundary_low_contrast_fraction", "boundary_min_contrast_db"],
        ascending=[False, True],
    ).iloc[0]
display(pd.read_csv(boundary_row["boundary_audit_path"]))
display(Image(filename=str(boundary_row["boundary_plot_path"]), width=1100))

A low boundary-contrast flag is not an automatic error and never moves an edge. Breathy, weak, or slowly decaying ALS speech may have low contrast. It prompts listening/manual inspection. Parameter optimality cannot be claimed from plots alone; if timing accuracy is a reported result, use a diagnosis-blind manual boundary reference subset and report onset/offset error and overlap by group.

In [ ]:
run_cli("segment-template")
segmentation_path = ROOT / yaml.safe_load(CONFIG.read_text(encoding="utf-8")).get(
    "data_freeze", {}
).get("segmentation_adjudication", "config/segmentation_adjudication.csv")
manual_path = ROOT / yaml.safe_load(CONFIG.read_text(encoding="utf-8")).get(
    "data_freeze", {}
).get("manual_segmentation_overrides", "config/manual_segmentation_overrides.csv")
segmentation_review = pd.read_csv(segmentation_path, keep_default_na=False)
from paper1_qc.segmentation import segmentation_pending_reviews

required_review = segmentation_review.loc[
    segmentation_review["review_required"].astype(str).str.lower().isin(["true", "1", "yes"])
].copy()
automatic_task_exclusions = segmentation_review.loc[
    segmentation_review["automatic_task_exclusion"].astype(str).str.lower().isin(
        ["true", "1", "yes"]
    )
].copy()
pending_review = segmentation_pending_reviews(segmentation_review)
display(required_review[[
    "file_name", "automatic_qc_status", "task_completed_as_instructed",
    "accepted_outlier", "review_reasons",
    "decision", "boundary_source", "reviewer", "review_date", "notes"
]])
display(Markdown(
    f"**Locked automatic task exclusions:** {len(automatic_task_exclusions)} "
    "(Task Completed as Instructed = NO)"
))
display(automatic_task_exclusions[[
    "file_name", "task_completed_as_instructed", "automatic_exclusion_reason",
    "decision", "boundary_source", "notes"
]])
segmentation_review_ready = stage_gate(
    "Segmentation review queue",
    pending_review.empty,
    [f"{len(pending_review)} required/incomplete reviews remain."],
    "Use the review widget below; then rerun this cell to update the gate.",
)

## Interactive recording review

The scrollable widget starts with **all recordings** and supports filtering and filename/ID search. It displays the original Silero plot, boundary audit, and an audio player. **Keep Silero + next** retains the automatic boundaries and advances. `KEEP + MANUAL` activates the optional boundary editor; enter one speech interval per line as `start_sec,end_sec`, preview it, and document why the correction was necessary. `EXCLUDE + NONE` removes the recording and requires a reason. Rows with `Task Completed as Instructed = NO` are shown but locked to exclusion. Manual editing changes speech boundaries only—it must not remove noise.

In [ ]:
from paper1_qc.config import load_config, resolve_executable
from paper1_qc.segmentation_review import launch_segmentation_review_widget

cfg = load_config(CONFIG)
automatic_intervals = read_table(STAGE / "bamboo_segmentation_intervals")
DEFAULT_REVIEWER = ""  # enter your name once, e.g. "Nevena Musikic"

review_widget = launch_segmentation_review_widget(
    summary=summary,
    automatic_intervals=automatic_intervals,
    review_path=segmentation_path,
    overrides_path=manual_path,
    default_reviewer=DEFAULT_REVIEWER,
    ffmpeg=resolve_executable(cfg["software"]["ffmpeg"], "ffmpeg"),
    ffprobe=resolve_executable(cfg["software"]["ffprobe"], "ffprobe"),
)
display(review_widget)

In [ ]:
# Run after using the widget. This reload is required because the widget saves to disk.
segmentation_review = pd.read_csv(segmentation_path, keep_default_na=False)
pending_review = segmentation_pending_reviews(segmentation_review)
review_progress = pd.DataFrame([{
    "total_recordings": len(segmentation_review),
    "mandatory_review_recordings": int(
        segmentation_review["review_required"].astype(str).str.lower().isin(["true", "1", "yes"]).sum()
    ),
    "pending_or_incomplete_reviews": len(pending_review),
    "automatic_task_exclusions": int(
        segmentation_review["automatic_task_exclusion"].astype(str).str.lower().isin(
            ["true", "1", "yes"]
        ).sum()
    ),
    "keep_auto": int((
        segmentation_review["decision"].eq("KEEP")
        & segmentation_review["boundary_source"].eq("AUTO")
    ).sum()),
    "keep_manual": int((
        segmentation_review["decision"].eq("KEEP")
        & segmentation_review["boundary_source"].eq("MANUAL")
    ).sum()),
    "exclude": int(segmentation_review["decision"].eq("EXCLUDE").sum()),
}])
display(review_progress)
display(pending_review[[
    "file_name", "automatic_qc_status", "task_completed_as_instructed",
    "review_reasons", "decision", "boundary_source", "reviewer",
    "review_date", "notes"
]])
save_table(review_progress, TABLES, "notebook_segmentation_review_progress")

In [ ]:
# Complete, auditable decision ledger before freezing.
review_audit = segmentation_review.copy()
as_bool = lambda value: (
    value if isinstance(value, bool)
    else str(value).strip().lower() in {"true", "1", "yes", "y"}
)
review_audit["review_required_bool"] = review_audit["review_required"].map(as_bool)
review_audit["automatic_task_exclusion_bool"] = (
    review_audit["automatic_task_exclusion"].map(as_bool)
)
review_audit["is_pending"] = review_audit["logical_recording_id"].astype(str).isin(
    set(pending_review["logical_recording_id"].astype(str))
)
decision_upper = review_audit["decision"].astype(str).str.strip().str.upper()
source_upper = review_audit["boundary_source"].astype(str).str.strip().str.upper()
review_audit["final_decision_category"] = np.select(
    [
        review_audit["automatic_task_exclusion_bool"]
        & decision_upper.eq("EXCLUDE") & source_upper.eq("NONE"),
        decision_upper.eq("KEEP") & source_upper.eq("AUTO")
        & ~review_audit["review_required_bool"] & ~review_audit["is_pending"],
        decision_upper.eq("KEEP") & source_upper.eq("AUTO")
        & review_audit["review_required_bool"] & ~review_audit["is_pending"],
        decision_upper.eq("KEEP") & source_upper.eq("MANUAL")
        & ~review_audit["is_pending"],
        decision_upper.eq("EXCLUDE") & source_upper.eq("NONE")
        & ~review_audit["is_pending"],
    ],
    [
        "SYSTEM_EXCLUDE_TASK_NOT_COMPLETED",
        "KEEP_AUTO_DEFAULT",
        "KEEP_AUTO_REVIEWED",
        "KEEP_MANUAL",
        "EXCLUDE_REVIEWED",
    ],
    default="UNRESOLVED",
)
review_audit["will_be_analysis_eligible"] = review_audit[
    "final_decision_category"
].isin(["KEEP_AUTO_DEFAULT", "KEEP_AUTO_REVIEWED", "KEEP_MANUAL"])
review_audit["boundary_provenance_after_freeze"] = np.select(
    [
        review_audit["final_decision_category"].isin(
            ["KEEP_AUTO_DEFAULT", "KEEP_AUTO_REVIEWED"]
        ),
        review_audit["final_decision_category"].eq("KEEP_MANUAL"),
        review_audit["final_decision_category"].str.startswith(
            ("EXCLUDE", "SYSTEM_EXCLUDE")
        ),
    ],
    [
        "automatic_silero",
        "manual_override",
        "not_used_excluded_recording",
    ],
    default="not_frozen_unresolved",
)
decision_audit_summary = (
    review_audit.groupby(
        [
            "final_decision_category",
            "will_be_analysis_eligible",
            "boundary_provenance_after_freeze",
        ],
        dropna=False,
    ).size().rename("logical_recordings").reset_index()
)
display(decision_audit_summary)
display(review_audit.loc[
    review_audit["final_decision_category"].eq("UNRESOLVED"),
    [
        "file_name", "automatic_qc_status", "task_completed_as_instructed",
        "review_reasons", "decision", "boundary_source", "reviewer",
        "review_date", "notes",
    ],
])
save_table(
    decision_audit_summary,
    TABLES,
    "notebook_segmentation_decision_audit_summary",
)
save_table(
    review_audit,
    TABLES,
    "notebook_segmentation_decision_audit_ledger",
)

In [ ]:
# Freeze only after the audit above contains zero UNRESOLVED rows.
# This cell captures full stdout/stderr instead of hiding the CLI failure cause.
RUN_SEGMENTATION_ADJUDICATION = False

if not RUN_SEGMENTATION_ADJUDICATION:
    print(
        "Segments not frozen. Set RUN_SEGMENTATION_ADJUDICATION=True only "
        "after pending_or_incomplete_reviews is zero."
    )
elif not pending_review.empty:
    print(
        f"BLOCKED: {len(pending_review)} pending/incomplete reviews remain. "
        "Return to the widget; do not freeze yet."
    )
else:
    command = [
        sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG),
        "segment-adjudicate",
    ]
    result = subprocess.run(
        command,
        cwd=ROOT,
        text=True,
        capture_output=True,
        check=False,
    )
    print("RETURN CODE:", result.returncode)
    print("\n========== STDOUT ==========")
    print(result.stdout or "(empty)")
    print("\n========== STDERR ==========")
    print(result.stderr or "(empty)")
    if result.returncode == 0:
        display(Markdown("**Segmentation freeze: PASS**"))
    else:
        display(Markdown(
            "**Segmentation freeze: BLOCKED.** Read STDERR above; do not continue."
        ))

In [ ]:
project_cfg = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
segmentation_freeze_version = project_cfg.get("segmentation_freeze", {}).get(
    "version",
    project_cfg.get("data_freeze", {}).get("version", "v1"),
)
SEGMENTATION_FREEZE = (
    MAIN_OUTPUTS / "01_SEGMENTATION_FREEZE" / str(segmentation_freeze_version)
)
REVIEWED_OUTPUT = OUTPUT / "01_segmentation_after_review"
decision_path = SEGMENTATION_FREEZE / "frozen_segmentation_decisions.csv"
interval_path = SEGMENTATION_FREEZE / "frozen_segmentation_intervals.csv"
reviewed_summary_path = (
    REVIEWED_OUTPUT / "segmentation" / "silero" / "summary"
    / "silero_after_review_summary.csv"
)
if decision_path.exists() and interval_path.exists() and reviewed_summary_path.exists():
    decisions = pd.read_csv(decision_path)
    frozen_intervals = pd.read_csv(interval_path)
    reviewed_summary = pd.read_csv(reviewed_summary_path)
    decision_summary = (
        decisions.groupby(
            ["automatic_qc_status", "review_required", "decision", "boundary_source"]
        ).size()
        .rename("logical_recordings").reset_index()
    )
    display(decision_summary)
    save_table(decision_summary, TABLES, "notebook_segmentation_decision_summary")
    reviewed_status = (
        reviewed_summary.groupby(
            ["final_review_status", "analysis_included"], dropna=False
        ).size().rename("logical_recordings").reset_index()
    )
    display(Markdown(
        "**Post-review contract:** accepted and flagged proceed; excluded do not."
    ))
    display(reviewed_status)
    save_table(
        reviewed_status,
        TABLES,
        "notebook_post_review_segmentation_status",
    )
    eligible_mask = decisions["segmentation_analysis_eligible"].map(
        lambda value: value if isinstance(value, bool)
        else str(value).strip().lower() in {"true", "1", "yes", "y"}
    )
    kept_ids = set(
        decisions.loc[eligible_mask, "logical_recording_id"].astype(str)
    )
    frozen_primary_ids = set(
        frozen_intervals.loc[
            frozen_intervals["profile"].eq("primary")
            & frozen_intervals["view"].eq("primary_speech")
            & frozen_intervals["duration_sec"].gt(0),
            "logical_recording_id",
        ].astype(str)
    )
    segmentation_ready = stage_gate(
        "Segmentation stage",
        decisions["decision"].isin(["KEEP", "EXCLUDE"]).all()
        and len(reviewed_summary) == len(decisions)
        and kept_ids.issubset(frozen_primary_ids),
        [
            f"{len(kept_ids - frozen_primary_ids)} KEEP recordings lack frozen primary speech."
        ] if not kept_ids.issubset(frozen_primary_ids) else [],
        "Open 02a Additive interference. Feature extraction reads only the versioned MAIN outputs freeze.",
    )
else:
    segmentation_ready = stage_gate(
        "Segmentation stage",
        False,
        [
            "Frozen decisions, frozen intervals, and/or "
            "outputs/01_segmentation_after_review do not exist."
        ],
        "Complete the widget review and run segment-adjudicate.",
    )